In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns


In [2]:
df=pd.read_csv("qoute_dataset.csv")

In [3]:
df.head()

,quote,Author
0,“The world as we have created it is a process ...,Albert Einstein
1,"“It is our choices, Harry, that show what we t...",J.K. Rowling
2,“There are only two ways to live your life. On...,Albert Einstein
3,"“The person, be it gentleman or lady, who has ...",Jane Austen
4,"“Imperfection is beauty, madness is genius and...",Marilyn Monroe


In [4]:
quotes=df["quote"]
quotes.head()

0    “The world as we have created it is a process ...
1    “It is our choices, Harry, that show what we t...
2    “There are only two ways to live your life. On...
3    “The person, be it gentleman or lady, who has ...
4    “Imperfection is beauty, madness is genius and...
Name: quote, dtype: object

In [5]:
quotes=quotes.str.lower()
quotes[0]


'“the world as we have created it is a process of our thinking. it cannot be changed without changing our thinking.”'

In [6]:
import string
translator=str.maketrans("","",string.punctuation)
quotes=df["quote"].apply(lambda x : x.translate(translator))

In [7]:
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer

In [8]:
vocab_size=8980

tokenize=Tokenizer(num_words=vocab_size)
tokenize.fit_on_texts(quotes)

In [9]:
word_index=tokenize.word_index
print(len(word_index))

8978


In [10]:
seq=tokenize.texts_to_sequences(quotes)

In [11]:
quotes[0]

'“The world as we have created it is a process of our thinking It cannot be changed without changing our thinking”'

In [12]:
seq[0]

[713,
 62,
 29,
 19,
 16,
 946,
 10,
 7,
 5,
 1156,
 8,
 70,
 293,
 10,
 145,
 12,
 809,
 104,
 752,
 70,
 2461]

In [13]:
x=[]
y=[]

In [14]:
for seq in seq:
    for i in range(1,len(seq)):
        input_seq=seq[:i]
        output_seq=seq[i]
        x.append(input_seq)
        y.append(output_seq)

In [15]:
x

[[713],
 [713, 62],
 [713, 62, 29],
 [713, 62, 29, 19],
 [713, 62, 29, 19, 16],
 [713, 62, 29, 19, 16, 946],
 [713, 62, 29, 19, 16, 946, 10],
 [713, 62, 29, 19, 16, 946, 10, 7],
 [713, 62, 29, 19, 16, 946, 10, 7, 5],
 [713, 62, 29, 19, 16, 946, 10, 7, 5, 1156],
 [713, 62, 29, 19, 16, 946, 10, 7, 5, 1156, 8],
 [713, 62, 29, 19, 16, 946, 10, 7, 5, 1156, 8, 70],
 [713, 62, 29, 19, 16, 946, 10, 7, 5, 1156, 8, 70, 293],
 [713, 62, 29, 19, 16, 946, 10, 7, 5, 1156, 8, 70, 293, 10],
 [713, 62, 29, 19, 16, 946, 10, 7, 5, 1156, 8, 70, 293, 10, 145],
 [713, 62, 29, 19, 16, 946, 10, 7, 5, 1156, 8, 70, 293, 10, 145, 12],
 [713, 62, 29, 19, 16, 946, 10, 7, 5, 1156, 8, 70, 293, 10, 145, 12, 809],
 [713, 62, 29, 19, 16, 946, 10, 7, 5, 1156, 8, 70, 293, 10, 145, 12, 809, 104],
 [713,
  62,
  29,
  19,
  16,
  946,
  10,
  7,
  5,
  1156,
  8,
  70,
  293,
  10,
  145,
  12,
  809,
  104,
  752],
 [713,
  62,
  29,
  19,
  16,
  946,
  10,
  7,
  5,
  1156,
  8,
  70,
  293,
  10,
  145,
  12,
  809,
  

In [16]:
max_len=max(len(x) for x in x)
print(max_len)


745


In [17]:
from tensorflow.keras.preprocessing.sequence import pad_sequences
X_padded=pad_sequences(x,maxlen=max_len,padding="pre")

In [18]:
X_padded

array([[   0,    0,    0, ...,    0,    0,  713],
       [   0,    0,    0, ...,    0,  713,   62],
       [   0,    0,    0, ...,  713,   62,   29],
       ...,
       [   0,    0,    0, ...,    9,   19, 1125],
       [   0,    0,    0, ...,   19, 1125,    3],
       [   0,    0,    0, ..., 1125,    3,  169]],
      shape=(85271, 745), dtype=int32)

In [19]:
y=np.array(y)

In [20]:
from tensorflow.keras.utils import to_categorical
y_one_hot=to_categorical(y,num_classes=vocab_size)

In [21]:
y.shape

(85271,)

In [22]:
y_one_hot.shape

(85271, 8980)

In [23]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM,Dense,Embedding,SimpleRNN

In [24]:
Embed_dim=50
rnn_units=128

In [25]:
rnn_model=Sequential()

rnn_model.add(
    Embedding(input_dim=vocab_size,output_dim=Embed_dim,input_length=max_len)
)

rnn_model.add(SimpleRNN( units=rnn_units))
rnn_model.add(Dense(units=vocab_size,activation="softmax"))

c:\Users\mansi\anaconda3\Lib\site-packages\keras\src\layers\core\embedding.py:123: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


In [26]:
rnn_model.compile(optimizer="adam",loss="categorical_crossentropy",metrics=["accuracy"])

In [27]:
rnn_model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ simple_rnn (SimpleRNN)          │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [28]:
lstm_model = Sequential([
    Embedding(
        input_dim=vocab_size,
        output_dim=Embed_dim
    ),
    LSTM(128),
    Dense(vocab_size, activation="softmax")
])

In [29]:
lstm_model.compile(optimizer="adam",loss="categorical_crossentropy",metrics=["accuracy"])

In [30]:
lstm_model.summary()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_1 (Embedding)         │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [31]:
epochs=10
batch_size=128


In [32]:
histor_rnn=rnn_model.fit(
    X_padded,y_one_hot,
    epochs=epochs,
    batch_size=batch_size,
    validation_split=0.1,
    )

Epoch 1/10
600/600 ━━━━━━━━━━━━━━━━━━━━ 227s 376ms/step - accuracy: 0.0466 - loss: 6.6883 - val_accuracy: 0.0648 - val_loss: 6.5017
Epoch 2/10
600/600 ━━━━━━━━━━━━━━━━━━━━ 285s 474ms/step - accuracy: 0.0825 - loss: 6.0551 - val_accuracy: 0.0951 - val_loss: 6.3002
Epoch 3/10
600/600 ━━━━━━━━━━━━━━━━━━━━ 293s 488ms/step - accuracy: 0.1060 - loss: 5.6911 - val_accuracy: 0.1041 - val_loss: 6.2678
Epoch 4/10
600/600 ━━━━━━━━━━━━━━━━━━━━ 341s 568ms/step - accuracy: 0.1223 - loss: 5.3884 - val_accuracy: 0.1101 - val_loss: 6.2953
Epoch 5/10
600/600 ━━━━━━━━━━━━━━━━━━━━ 359s 598ms/step - accuracy: 0.1367 - loss: 5.1146 - val_accuracy: 0.1093 - val_loss: 6.3626
Epoch 6/10
600/600 ━━━━━━━━━━━━━━━━━━━━ 341s 568ms/step - accuracy: 0.1524 - loss: 4.8658 - val_accuracy: 0.1126 - val_loss: 6.4179
Epoch 7/10
600/600 ━━━━━━━━━━━━━━━━━━━━ 173s 287ms/step - accuracy: 0.1677 - loss: 4.6341 - val_accuracy: 0.1137 - val_loss: 6.4967
Epoch 8/10
600/600 ━━━━━━━━━━━━━━━━━━━━ 307s 512ms/step - accuracy: 0.1882 -

In [33]:
lstm_model.summary()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_1 (Embedding)         │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [ ]:
from tensorflow.keras.callbacks import EarlyStopping

early_stop = EarlyStopping(
    monitor='val_accuracy',
    patience=5,
    mode='max',
    restore_best_weights=True
)

history = model.fit(
    X_padded,
    y_one_hot,
    epochs=100,
    batch_size=128,
    validation_split=0.1,
    callbacks=[early_stop]
)

Epoch 1/100
600/600 ━━━━━━━━━━━━━━━━━━━━ 1574s 3s/step - accuracy: 0.0398 - loss: 6.7465 - val_accuracy: 0.0436 - val_loss: 6.6969
Epoch 2/100
600/600 ━━━━━━━━━━━━━━━━━━━━ 5487s 9s/step - accuracy: 0.0589 - loss: 6.3183 - val_accuracy: 0.0670 - val_loss: 6.5427
Epoch 3/100
600/600 ━━━━━━━━━━━━━━━━━━━━ 1439s 2s/step - accuracy: 0.0814 - loss: 6.0469 - val_accuracy: 0.0881 - val_loss: 6.4547
Epoch 4/100
600/600 ━━━━━━━━━━━━━━━━━━━━ 819s 1s/step - accuracy: 0.0972 - loss: 5.8314 - val_accuracy: 0.0950 - val_loss: 6.4232
Epoch 5/100
600/600 ━━━━━━━━━━━━━━━━━━━━ 744s 1s/step - accuracy: 0.1098 - loss: 5.6493 - val_accuracy: 0.0996 - val_loss: 6.4110
Epoch 6/100
600/600 ━━━━━━━━━━━━━━━━━━━━ 738s 1s/step - accuracy: 0.1202 - loss: 5.4738 - val_accuracy: 0.1047 - val_loss: 6.4108
Epoch 7/100
600/600 ━━━━━━━━━━━━━━━━━━━━ 1576s 3s/step - accuracy: 0.1299 - loss: 5.3159 - val_accuracy: 0.1076 - val_loss: 6.4306
Epoch 8/100
600/600 ━━━━━━━━━━━━━━━━━━━━ 4309s 7s/step - accuracy: 0.1372 - loss: 5.17

KeyboardInterrupt: 

In [ ]:
lstm_model.save("lstm_model.h5")

In [ ]:
index_to_word={}
for word,index in word_index.items():
    index_to_word[index]=word


In [ ]:
index_to_words

In [ ]:
def Predictor(model,tokenizer,text,max_len)
    text=text.lower()
    sequence=tokenizer.text_to_sequences([text])
    padding=pad_sequence([sequence],maxlen=max_len,padding="pre")

    predict=model.predict(sequence,verbose=0)
    pred_index=np.argmax(predict)
    return index_to_word[pred_index]

In [ ]:
seed_text="life is"
predict_next=Predictor(lstm_model,tokenizer,seed_text,max_len)
print(next_word)

In [ ]:
def generateSentence(model,tokenizer,max_len,n_words):
    for _ in range(n_words):
        next_word=Predictor(model,tokenizer,seed_text,max_len)
        if next_word=="":
            break
        seed_text=""+next_word
    return seed_text

In [ ]:
import pickle
with open("tokenizer.pkl","wb") as f:
    pickle.dump()